In [2]:
# CHUNKED FAST BATCH: runs flookup in large chunks with progress printing
from __future__ import annotations
import csv, os, re, sys, time
from typing import Dict, List, Tuple

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from src.disambiguation import tokenize, PRESERVE_TOKEN, PUNCTUATIONS
import src.foma_adapter as fa  # uses your patched, explicit-path version

TSV_PATH = "example_sentences.tsv"
OUT_OJ = "oj_opd.txt"
OUT_EN = "en_opd.txt"

FST_PATH = os.path.abspath(os.path.join(os.getcwd(), "../fst/ojibwe7.fomabin"))
if not os.path.exists(FST_PATH):
    FST_PATH = os.path.abspath(os.path.join(os.getcwd(), "../data/fst/ojibwe7.fomabin"))
print("[info] Using FST:", FST_PATH)

WORD_RE = re.compile(r"^[\wʼ’'\-–]+$", re.UNICODE)

def normalize_orth(s: str) -> str:
    return (s.replace("’", "'").replace("‘", "'")
             .replace("–", "-").replace("—", "-")
             .replace("\u200b", ""))

def filter_word_tokens(tokens: List[str]) -> List[str]:
    out = []
    for t in tokens:
        if t == PRESERVE_TOKEN:
            continue
        if len(t) == 1 and t in PUNCTUATIONS:
            continue
        if WORD_RE.match(t):
            out.append(t)
    return out

# Load rows
if not os.path.exists(TSV_PATH):
    raise FileNotFoundError(f"Put {TSV_PATH} next to the notebook.")
rows: List[Tuple[str, str]] = []
with open(TSV_PATH, "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        oj = (row.get("Ojibwe") or "").strip()
        en = (row.get("English") or "").strip()
        if oj:
            rows.append((oj, en))
print(f"[info] Loaded rows: {len(rows)}")

# Gather unique tokens
unique_tokens: List[str] = []
seen = set()
per_row_tokens: List[List[str]] = []
for oj, _ in rows:
    toks = filter_word_tokens(tokenize(normalize_orth(oj)))
    per_row_tokens.append(toks)
    for t in toks:
        if t not in seen:
            seen.add(t); unique_tokens.append(t)
print(f"[info] Unique tokens: {len(unique_tokens)}")

# Chunked flookup
CHUNK_SIZE = 5000  # tune if needed
analysis_map: Dict[str, List[str]] = {}
n = len(unique_tokens)
start = time.time()
for i in range(0, n, CHUNK_SIZE):
    chunk = unique_tokens[i:i+CHUNK_SIZE]
    t0 = time.time()
    res = fa.flookup(chunk, bin_path=FST_PATH)  # explicit path (no fallback)
    for j, token in enumerate(chunk):
        analyses = res[j].get("fst_analyses", []) if j < len(res) else []
        analysis_map[token] = analyses
    dt = time.time() - t0
    print(f"[batch] {i+len(chunk)}/{n} tokens processed in {dt:.2f}s "
          f"({(i+len(chunk))/n:.1%} done)")

print(f"[done] Token analyses built in {time.time()-start:.2f}s")

# Evaluate rows
kept_pairs: List[Tuple[str, str]] = []
failed_coverage = 0
failed_no_ambiguity = 0

for (oj, en), toks in zip(rows, per_row_tokens):
    if not toks:
        failed_coverage += 1
        continue
    all_parsed = all(len(analysis_map.get(t, [])) >= 1 for t in toks)
    if not all_parsed:
        failed_coverage += 1
        continue
    has_ambig = any(len(analysis_map.get(t, [])) >= 2 for t in toks)
    if not has_ambig:
        failed_no_ambiguity += 1
        continue
    kept_pairs.append((oj, en))

# Write outputs
with open(OUT_OJ, "w", encoding="utf-8") as fo:
    fo.write("\n".join(oj for oj, _ in kept_pairs) + ("\n" if kept_pairs else ""))

with open(OUT_EN, "w", encoding="utf-8") as fe:
    fe.write("\n".join(en for _, en in kept_pairs) + ("\n" if kept_pairs else ""))

print("=== Summary ===")
print(f"Total rows read:           {len(rows)}")
print(f"Kept (coverage+ambiguity): {len(kept_pairs)}")
print(f"Failed coverage:           {failed_coverage}")
print(f"Failed (no ambiguity):     {failed_no_ambiguity}")
print(f"Wrote: {OUT_OJ}  and  {OUT_EN}")


[info] Using FST: /Users/matthias/ELF-Lab Repos/Ojibwe_Constraint_Grammar/data/fst/ojibwe7.fomabin
[info] Loaded rows: 4645
[info] Unique tokens: 9811
[batch] 5000/9811 tokens processed in 0.20s (51.0% done)
[batch] 9811/9811 tokens processed in 0.25s (100.0% done)
[done] Token analyses built in 0.45s
=== Summary ===
Total rows read:           4645
Kept (coverage+ambiguity): 2446
Failed coverage:           867
Failed (no ambiguity):     1332
Wrote: oj_opd.txt  and  en_opd.txt


In [ ]:
# Make 50-line subsets of oj_opd.txt and en_opd.txt

def take_first_n(infile: str, outfile: str, n: int = 50):
    with open(infile, "r", encoding="utf-8") as f:
        lines = [line.rstrip("\n") for line in f][:n]
    with open(outfile, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + ("\n" if lines else ""))

take_first_n("../parallel/oj_opd.txt", "../parallel/oj_1000.txt", 1000)
take_first_n("../parallel/en_opd.txt", "../parallel/en_1000.txt", 1000)

Created oj_50.txt and en_50.txt with first 50 lines each.
